# Exploratory Data Analysis — ASVspoof 2019 (LA)

**DA 515 Final Project | Yves Alain Iragena**

This notebook explores the ASVspoof 2019 dataset: class balance, waveform properties, and feature distributions.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from datasets import load_dataset

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

## 1. Load Dataset

In [ ]:
dataset = load_dataset('asvspoof/asvspoof2019', 'LA', trust_remote_code=True)
print(dataset)

train_df = dataset['train'].to_pandas()
dev_df   = dataset['validation'].to_pandas()
eval_df  = dataset['eval'].to_pandas()

print(f'\nTrain: {len(train_df)} | Dev: {len(dev_df)} | Eval: {len(eval_df)}')

## 2. Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (df, name) in zip(axes, [(train_df, 'Train'), (dev_df, 'Dev'), (eval_df, 'Eval')]):
    counts = df['label'].value_counts().rename({0: 'Spoof', 1: 'Bonafide'})
    counts.plot(kind='bar', ax=ax, color=['#e74c3c', '#2ecc71'], edgecolor='black')
    ax.set_title(f'{name} Set')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=0)
    for p in ax.patches:
        ax.annotate(f'{int(p.get_height()):,}', (p.get_x() + 0.05, p.get_height() + 50))

plt.suptitle('Class Distribution Across Splits', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Spoofing Attack Types (TTS/VC Systems)

In [ ]:
if 'system_id' in train_df.columns:
    attack_counts = train_df[train_df['label'] == 0]['system_id'].value_counts()
    fig, ax = plt.subplots(figsize=(10, 4))
    attack_counts.plot(kind='bar', ax=ax, color='#e74c3c', edgecolor='black')
    ax.set_title('Spoofing Attack Types in Training Set')
    ax.set_xlabel('System ID (TTS/VC Algorithm)')
    ax.set_ylabel('Count')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('system_id column not available in this version of the dataset.')

## 4. Waveform & Duration Analysis

In [ ]:
SR = 16000
N_SAMPLE = 200

sample_df = train_df.sample(N_SAMPLE, random_state=42)
durations = []

for _, row in sample_df.iterrows():
    audio = row['audio']
    dur = len(audio['array']) / audio['sampling_rate']
    durations.append({'duration': dur, 'label': 'Bonafide' if row['label'] == 1 else 'Spoof'})

dur_df = pd.DataFrame(durations)

fig, ax = plt.subplots(figsize=(9, 4))
for label, color in [('Bonafide', '#2ecc71'), ('Spoof', '#e74c3c')]:
    subset = dur_df[dur_df['label'] == label]['duration']
    ax.hist(subset, bins=30, alpha=0.6, color=color, label=label, edgecolor='black')

ax.set_xlabel('Duration (seconds)')
ax.set_ylabel('Count')
ax.set_title('Audio Duration Distribution (200-sample subset)')
ax.legend()
plt.tight_layout()
plt.show()

print(dur_df.groupby('label')['duration'].describe().round(2))

## 5. Waveform Comparison: Bonafide vs. Spoof

In [ ]:
bonafide_sample = train_df[train_df['label'] == 1].iloc[0]
spoof_sample    = train_df[train_df['label'] == 0].iloc[0]

fig, axes = plt.subplots(2, 1, figsize=(12, 5))
for ax, row, title in [
    (axes[0], bonafide_sample, 'Bonafide Speech'),
    (axes[1], spoof_sample,    'Synthetic (Spoofed) Speech'),
]:
    waveform = np.array(row['audio']['array'])
    sr = row['audio']['sampling_rate']
    t = np.linspace(0, len(waveform)/sr, len(waveform))
    ax.plot(t, waveform, linewidth=0.5, alpha=0.8)
    ax.set_title(title)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

## 6. Spectrogram Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

for row_idx, (row, label) in enumerate([
    (bonafide_sample, 'Bonafide'),
    (spoof_sample, 'Spoof'),
]):
    waveform = np.array(row['audio']['array'], dtype=np.float32)
    sr = row['audio']['sampling_rate']

    # MFCC
    mfcc = librosa.feature.mfcc(y=waveform, sr=sr, n_mfcc=40)
    librosa.display.specshow(mfcc, sr=sr, x_axis='time', ax=axes[row_idx, 0])
    axes[row_idx, 0].set_title(f'{label} — MFCC')
    axes[row_idx, 0].set_ylabel('MFCC Coefficient')

    # Log-Mel Spectrogram
    mel = librosa.feature.melspectrogram(y=waveform, sr=sr, n_mels=80)
    log_mel = librosa.power_to_db(mel, ref=np.max)
    img = librosa.display.specshow(log_mel, sr=sr, x_axis='time', y_axis='mel', ax=axes[row_idx, 1])
    axes[row_idx, 1].set_title(f'{label} — Log-Mel Spectrogram')
    plt.colorbar(img, ax=axes[row_idx, 1], format='%+2.0f dB')

plt.suptitle('Feature Comparison: Bonafide vs. Synthetic Speech', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Key Observations

- **Class imbalance:** The training set has significantly more spoof samples than bonafide, motivating the use of EER as primary metric over accuracy.
- **Spectral artifacts:** TTS-generated speech often shows overly regular harmonic structures and lacks the micro-variation present in genuine speech.
- **Feature motivation:** MFCC captures physiological resonances; LFCC preserves high-frequency information where synthesis artifacts are most visible — both are needed.